# 01 — Audit Dataset PhiUSIIL (Proyek TTKI P1)

**Tujuan notebook ini:** mengenal data sebelum ada model apa pun. Notebook ini tidak melatih model dan tidak menyentuh data uji.
Ada enam pertanyaan audit:

1. Versi data dan kapan diunduh
2. Ukuran data dan arti label
3. URL berulang dan konflik label
4. Kolom mana yang URL-only, butuh halaman, atau belum jelas
5. Pengelompokan registered domain dan ukuran grupnya
6. Apakah pembagian berbasis grup (grouped split) layak dilakukan

Hasil ringkasan disimpan ke `results/data_audit.csv`, `results/column_classification.csv`, dan `results/split_audit.csv`.

## 0. Persiapan

In [1]:
import os, hashlib, platform, datetime, json
import numpy as np, pandas as pd, sklearn, tldextract
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

SEED = 42
DOWNLOAD_DATE = "2026-11-12"   # tanggal unduh menurut proposal
CANDIDATES = [
    "../data/PhiUSIIL_Phishing_URL_Dataset.csv",
    "data/PhiUSIIL_Phishing_URL_Dataset.csv",
    "/mnt/user-data/uploads/PhiUSIIL_Phishing_URL_Dataset.csv",
]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
RESULTS_DIR = "../results" if os.path.isdir("../results") else "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

audit = {}   # semua angka audit dikumpulkan di sini lalu disimpan ke CSV
print("Python", platform.python_version(), "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__, "| tldextract", tldextract.__version__)
print("Data:", DATA_PATH)

Python 3.12.3 | pandas 3.0.6 | sklearn 1.9.1 | tldextract 5.4.0
Data: /mnt/user-data/uploads/PhiUSIIL_Phishing_URL_Dataset.csv


## 1. Versi data
Sidik jari file (SHA-256) dicatat supaya anggota kelompok lain bisa memastikan mereka memakai file yang sama persis.

In [2]:
sha = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
audit["file_name"] = os.path.basename(DATA_PATH)
audit["file_size_bytes"] = os.path.getsize(DATA_PATH)
audit["file_sha256"] = sha
audit["download_date"] = DOWNLOAD_DATE
print(audit)

{'file_name': 'PhiUSIIL_Phishing_URL_Dataset.csv', 'file_size_bytes': 56854345, 'file_sha256': 'a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6', 'download_date': '2026-11-12'}


## 2. Ukuran data dan arti label
Pada dataset asli, `label = 1` berarti **legitimate** dan `label = 0` berarti **phishing**.
Proposal membalik label menjadi `y = 1` untuk phishing supaya phishing menjadi kelas positif
(F1 dan recall kemudian mengukur kemampuan menangkap phishing).

In [3]:
df = pd.read_csv(DATA_PATH)
audit["n_rows"], audit["n_cols"] = df.shape
print("Bentuk data:", df.shape)

print("\nDistribusi label asli:")
print(df["label"].value_counts().sort_index())

# pembalikan label: y=1 -> phishing
df["y"] = 1 - df["label"]
audit["n_legitimate"] = int((df["label"] == 1).sum())
audit["n_phishing"] = int((df["label"] == 0).sum())
audit["phishing_ratio"] = round(df["y"].mean(), 4)
assert set(df["label"].unique()) == {0, 1}
print("\nSetelah dibalik, y=1 (phishing):", int(df['y'].sum()), "baris =", f"{df['y'].mean():.1%}")

audit["n_missing_cells"] = int(df.drop(columns="y").isna().sum().sum())
audit["n_exact_duplicate_rows"] = int(df.drop(columns=["FILENAME"]).duplicated().sum())
print("Sel kosong:", audit["n_missing_cells"], "| baris identik (tanpa FILENAME):", audit["n_exact_duplicate_rows"])

Bentuk data: (235795, 56)

Distribusi label asli:
label
0    100945
1    134850
Name: count, dtype: int64

Setelah dibalik, y=1 (phishing): 100945 baris = 42.8%


Sel kosong: 0 | baris identik (tanpa FILENAME): 0


## 3. URL berulang dan konflik label
Baris identik dan **URL yang sama** adalah dua hal berbeda. Dua baris bisa punya isi halaman yang berbeda
tetapi URL yang sama. Konflik label berarti URL yang sama muncul sebagai phishing sekaligus legitimate.

In [4]:
audit["n_unique_url"] = int(df["URL"].nunique())
dup_url_rows = int(df["URL"].duplicated(keep=False).sum())
audit["n_rows_with_repeated_url"] = dup_url_rows

lab_per_url = df.groupby("URL")["label"].nunique()
conflict_urls = lab_per_url[lab_per_url > 1].index
audit["n_urls_with_label_conflict"] = int(len(conflict_urls))
audit["n_rows_in_label_conflict"] = int(df["URL"].isin(conflict_urls).sum())
audit["n_duplicate_filename"] = int(df["FILENAME"].duplicated().sum())

for k in ["n_unique_url","n_rows_with_repeated_url","n_urls_with_label_conflict","n_rows_in_label_conflict","n_duplicate_filename"]:
    print(f"{k:32s}", audit[k])

if len(conflict_urls):
    print("\nContoh URL berkonflik:")
    print(df[df['URL'].isin(conflict_urls)][["URL","label"]].sort_values("URL").head(10).to_string())

n_unique_url                     235370
n_rows_with_repeated_url         850
n_urls_with_label_conflict       0
n_rows_in_label_conflict         0
n_duplicate_filename             0


## 4. Klasifikasi kolom: URL-only, butuh halaman, atau belum jelas
Dari 56 kolom: `FILENAME` (penanda), `label` (target), dan **54 kolom lain**. Empat di antaranya berupa teks
(`URL`, `Domain`, `TLD`, `Title`) sehingga **50 kolom numerik** yang bisa jadi fitur model.

Pembagian di bawah adalah **draf awal dari kelompok**, bukan fakta dari penulis dataset:

- **F-URL**: bisa dihitung hanya dari teks URL.
- **F-Page**: butuh membuka dan membaca halaman web.
- **Belum jelas**: asal pembentukannya belum terverifikasi (misalnya dihitung memakai statistik yang dipelajari dari seluruh dataset). Dikeluarkan dari analisis utama sesuai proposal.

In [5]:
TEXT_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]

F_URL = ["URLLength","DomainLength","IsDomainIP","TLDLength","NoOfSubDomain","HasObfuscation",
         "NoOfObfuscatedChar","ObfuscationRatio","NoOfLettersInURL","LetterRatioInURL",
         "NoOfDegitsInURL","DegitRatioInURL","NoOfEqualsInURL","NoOfQMarkInURL","NoOfAmpersandInURL",
         "NoOfOtherSpecialCharsInURL","SpacialCharRatioInURL","IsHTTPS"]

UNCLEAR = ["URLSimilarityIndex","CharContinuationRate","TLDLegitimateProb","URLCharProb"]

F_PAGE = ["LineOfCode","LargestLineLength","HasTitle","DomainTitleMatchScore","URLTitleMatchScore",
          "HasFavicon","Robots","IsResponsive","NoOfURLRedirect","NoOfSelfRedirect","HasDescription",
          "NoOfPopup","NoOfiFrame","HasExternalFormSubmit","HasSocialNet","HasSubmitButton",
          "HasHiddenFields","HasPasswordField","Bank","Pay","Crypto","HasCopyrightInfo",
          "NoOfImage","NoOfCSS","NoOfJS","NoOfSelfRef","NoOfEmptyRef","NoOfExternalRef"]

numeric = [c for c in df.columns if c not in TEXT_COLS + ["label","y"]]
assigned = F_URL + UNCLEAR + F_PAGE
print("Kolom numerik:", len(numeric), "| terklasifikasi:", len(assigned))
print("Belum terklasifikasi:", sorted(set(numeric) - set(assigned)))
print("Terklasifikasi tapi tidak ada di data:", sorted(set(assigned) - set(numeric)))
assert sorted(numeric) == sorted(assigned) and len(set(assigned)) == len(assigned)

Kolom numerik: 50 | terklasifikasi: 50
Belum terklasifikasi: []
Terklasifikasi tapi tidak ada di data: []


**Verifikasi empiris.** Klaim "F-URL hanya bergantung pada teks URL" bisa diuji: kalau kolom itu benar-benar fitur URL,
kita harus bisa menghitung ulang nilainya dari kolom `URL`. Kecocokan tinggi mendukung klasifikasi; ketidakcocokan berarti
definisinya berbeda dari dugaan kita.

In [6]:
import re
u = df["URL"]
checks = {
  "URLLength":        u.str.len(),
  "DomainLength":     df["Domain"].str.len(),
  "TLDLength":        df["TLD"].str.len(),
  "NoOfLettersInURL": u.str.count(r"[A-Za-z]"),
  "NoOfDegitsInURL":  u.str.count(r"[0-9]"),
  "NoOfEqualsInURL":  u.str.count("="),
  "NoOfQMarkInURL":   u.str.count(r"\?"),
  "NoOfAmpersandInURL": u.str.count("&"),
  "IsHTTPS":          u.str.lower().str.startswith("https").astype(int),
}
rows = []
for col, recomputed in checks.items():
    match = (recomputed.values == df[col].values).mean()
    rows.append({"kolom": col, "cocok_dgn_hitung_ulang": round(match, 4)})
verif = pd.DataFrame(rows); print(verif.to_string(index=False))

             kolom  cocok_dgn_hitung_ulang
         URLLength                  0.2063
      DomainLength                  1.0000
         TLDLength                  1.0000
  NoOfLettersInURL                  0.0000
   NoOfDegitsInURL                  0.9949
   NoOfEqualsInURL                  0.9989
    NoOfQMarkInURL                  0.9999
NoOfAmpersandInURL                  0.9865
           IsHTTPS                  0.9979


**Catatan hasil verifikasi.** `DomainLength`, `TLDLength` cocok 100%, dan jumlah digit, `=`, `?`, `&`, serta `IsHTTPS` cocok di atas 98%.
Namun `URLLength` hampir selalu berbeda 1 dari `len(URL)` (selisih -1 pada sekitar 79% baris), dan `NoOfLettersInURL` tidak pernah sama
dengan hitungan huruf pada URL utuh (selisihnya sekitar 7–9 huruf, sesuai dugaan awalan seperti `https` dan `www` tidak ikut dihitung).
Artinya fitur ini memang diturunkan dari teks URL, tetapi **definisi persisnya berbeda dari hitungan sederhana**. Cek definisinya
di paper PhiUSIIL atau repo referensi sebelum menuliskannya di bagian Metode.

In [7]:
# Tabel klasifikasi + ringkasan korelasi sederhana dengan target (hanya deskriptif, bukan seleksi fitur)
group_of = {**{c:"F-URL" for c in F_URL}, **{c:"Belum jelas" for c in UNCLEAR}, **{c:"F-Page" for c in F_PAGE}}
col_tbl = pd.DataFrame({
    "kolom": numeric,
    "kelompok": [group_of[c] for c in numeric],
    "n_unik": [df[c].nunique() for c in numeric],
    "min": [df[c].min() for c in numeric],
    "max": [df[c].max() for c in numeric],
})
col_tbl["catatan"] = ""
col_tbl.loc[col_tbl.kolom.isin(UNCLEAR), "catatan"] = "asal pembentukan belum diverifikasi; keluar dari analisis utama"
col_tbl.loc[col_tbl.kolom.isin(["NoOfURLRedirect","NoOfSelfRedirect"]), "catatan"] = "butuh permintaan HTTP"
col_tbl.to_csv(f"{RESULTS_DIR}/column_classification.csv", index=False)
audit["n_numeric_features"] = len(numeric)
audit["n_F_URL"], audit["n_F_Page"], audit["n_unclear"] = len(F_URL), len(F_PAGE), len(UNCLEAR)
print(col_tbl["kelompok"].value_counts())

kelompok
F-Page         28
F-URL          18
Belum jelas     4
Name: count, dtype: int64


## 5. Pengelompokan registered domain
**Registered domain (eTLD+1)** adalah bagian domain yang bisa didaftarkan seseorang, misalnya
`www.voicefmradio.co.uk` menjadi `voicefmradio.co.uk`. Ini dipakai sebagai kunci grup: semua URL dengan registered domain
yang sama harus masuk ke sisi yang sama (train atau test).

Ada satu keputusan yang harus dicatat: **platform hosting** seperti `github.io` atau `blogspot.com`. Secara default,
`tldextract` menganggap `github.io` sebagai suffix publik biasa, sehingga `a.github.io` dan `b.github.io` dihitung sebagai
registered domain yang sama. Daftar PSL "private" memperlakukannya sebagai domain terpisah. Kita hitung keduanya dan bandingkan.

In [8]:
ext_default = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=False)
ext_private = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=True)

def reg_domain(ext, url):
    r = ext(url)
    if r.suffix:                      # kasus normal
        return f"{r.domain}.{r.suffix}".lower()
    return r.domain.lower()           # IP atau host tanpa suffix

urls = df["URL"].tolist()
df["reg_domain"]         = [reg_domain(ext_default, x) for x in urls]
df["reg_domain_private"] = [reg_domain(ext_private, x) for x in urls]
audit["n_empty_suffix_rows"] = int(sum(1 for x in urls if not ext_default(x).suffix))
print("Contoh:"); print(df[["URL","reg_domain","reg_domain_private"]].sample(5, random_state=SEED).to_string(index=False))

Contoh:
                                                                                                                                                                                                                                                                                                                                                                                              URL                   reg_domain           reg_domain_private
                                                                                                                                                                                                                                                                                                                                                                        https://www.northcm.ac.th                northcm.ac.th                northcm.ac.th
                                                                                                        

In [9]:
def group_summary(col):
    size = df.groupby(col)["y"].transform("size")
    gs = df.groupby(col).size()
    mixed = df.groupby(col)["y"].nunique()
    return {
        "n_groups": int(gs.shape[0]),
        "n_singleton_groups": int((gs == 1).sum()),
        "n_rows_in_groups_gt100": int((size > 100).sum()),
        "largest_group_size": int(gs.max()),
        "n_groups_with_both_labels": int((mixed > 1).sum()),
        "n_rows_in_mixed_label_groups": int(df[col].isin(mixed[mixed>1].index).sum()),
    }
for name, col in [("default", "reg_domain"), ("private_psl", "reg_domain_private")]:
    s = group_summary(col)
    for k, v in s.items(): audit[f"{name}_{k}"] = v
print(pd.DataFrame({"default": group_summary("reg_domain"), "private_psl": group_summary("reg_domain_private")}))
print("\nPerbandingan dengan angka di proposal: 175.509 grup, 169.496 singleton, 40.028 baris di grup > 100")

                              default  private_psl
n_groups                       175509       197709
n_singleton_groups             169496       190071
n_rows_in_groups_gt100          40028        15849
largest_group_size               5754         3097
n_groups_with_both_labels         120          106
n_rows_in_mixed_label_groups     1374          607

Perbandingan dengan angka di proposal: 175.509 grup, 169.496 singleton, 40.028 baris di grup > 100


In [10]:
top = (df.groupby("reg_domain")
         .agg(n=("y","size"), n_phishing=("y","sum"))
         .assign(phishing_ratio=lambda d: (d.n_phishing/d.n).round(3))
         .sort_values("n", ascending=False).head(15))
print("15 grup terbesar (registered domain default):"); print(top)

15 grup terbesar (registered domain default):
                     n  n_phishing  phishing_ratio
reg_domain                                        
web.app           5754        5754           1.000
firebaseapp.com   5594        5594           1.000
repl.co           3754        3754           1.000
weeblysite.com    3097        3097           1.000
ipfs.io           1559        1559           1.000
workers.dev       1438        1438           1.000
square.site       1199        1199           1.000
dweb.link          975         975           1.000
xsph.ru            928         928           1.000
pantheonsite.io    900         900           1.000
google.com         782         782           1.000
godaddysites.com   781         781           1.000
duckdns.org        643         642           0.998
webwave.dev        567         567           1.000
weebly.com         555         555           1.000


Pertanyaan lanjutan yang penting: apakah grup besar itu **platform hosting** (banyak pemilik berbeda di satu domain)
atau satu pemilik? Itu mempengaruhi tafsir "domain tidak pernah terlihat". Keputusan kebijakan (default vs private PSL)
sebaiknya diambil kelompok setelah melihat tabel ini, dan dicatat di `configs/experiment_config.json`.

## 6. Kelayakan grouped split
Pertanyaannya: kalau kita bagi per grup, apakah masih bisa dapat (a) test sekitar 20%, (b) 5 fold pada development set,
dan (c) kedua kelas ada di setiap fold dengan proporsi wajar? Di sini kita hanya **memeriksa ukuran dan proporsi**,
tidak ada model, dan test tidak dipakai untuk apa pun.

Pada pembagian grouped, dua kondisi harus nol: grup yang muncul di train dan test sekaligus.

In [11]:
y = df["y"].values
groups = df["reg_domain"].values
rows = []

def record(protocol, part, idx, parent_idx=None):
    n = len(idx); p = int(y[idx].sum())
    rows.append({"protokol": protocol, "bagian": part, "n_baris": n,
                 "n_phishing": p, "rasio_phishing": round(p/n, 4),
                 "n_grup": len(set(groups[idx])),
                 "persen_dari_total": round(100*n/len(df), 2)})

# --- Protokol random stratified (skenario A-B) ---
rng_all = np.arange(len(df))
skf_out = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_r, test_r = next(skf_out.split(rng_all, y))
record("random", "test_final", test_r)
record("random", "development", dev_r)
for i, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(dev_r, y[dev_r])):
    record("random", f"cv_fold{i+1}_val", dev_r[va])
overlap_random = len(set(groups[dev_r]) & set(groups[test_r]))

# --- Protokol registered-domain-disjoint (skenario C-D) ---
sgk_out = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
dev_g, test_g = next(sgk_out.split(rng_all, y, groups))
record("grouped", "test_final", test_g)
record("grouped", "development", dev_g)
overlap_grouped = len(set(groups[dev_g]) & set(groups[test_g]))
cv_overlaps = []
for i, (tr, va) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(dev_g, y[dev_g], groups[dev_g])):
    record("grouped", f"cv_fold{i+1}_val", dev_g[va])
    cv_overlaps.append(len(set(groups[dev_g[tr]]) & set(groups[dev_g[va]])))

split_audit = pd.DataFrame(rows)
split_audit.to_csv(f"{RESULTS_DIR}/split_audit.csv", index=False)
audit["overlap_groups_random_dev_test"] = overlap_random
audit["overlap_groups_grouped_dev_test"] = overlap_grouped
audit["overlap_groups_grouped_cv_max"] = max(cv_overlaps)
print(split_audit.to_string(index=False))
print("\nRegistered domain yang muncul di dev DAN test -> random:", overlap_random, "| grouped:", overlap_grouped)
print("Overlap maksimum train-val pada CV grouped:", max(cv_overlaps))

protokol       bagian  n_baris  n_phishing  rasio_phishing  n_grup  persen_dari_total
  random   test_final    47159       20189          0.4281   36818               20.0
  random  development   188636       80756          0.4281  141588               80.0
  random cv_fold1_val    37728       16152          0.4281   29757               16.0
  random cv_fold2_val    37727       16151          0.4281   29680               16.0
  random cv_fold3_val    37727       16151          0.4281   29774               16.0
  random cv_fold4_val    37727       16151          0.4281   29678               16.0
  random cv_fold5_val    37727       16151          0.4281   29758               16.0
 grouped   test_final    47159       20189          0.4281   35098               20.0
 grouped  development   188636       80756          0.4281  140411               80.0
 grouped cv_fold1_val    37728       16152          0.4281   28078               16.0
 grouped cv_fold2_val    37728       16151          0.

**Cara membaca tabel di atas.** Pada protokol random, ada ribuan registered domain yang muncul di development dan test sekaligus;
inilah yang ingin diukur dampaknya. Pada protokol grouped, angka overlap harus 0. Perhatikan juga `persen_dari_total` dan
`rasio_phishing` pada fold grouped: kalau ada fold yang menyimpang jauh dari 20% atau dari rasio phishing keseluruhan,
penyebab biasanya satu grup yang sangat besar.

## 7. Simpan ringkasan audit

In [12]:
out = pd.DataFrame({"metrik": list(audit.keys()), "nilai": list(audit.values())})
out.to_csv(f"{RESULTS_DIR}/data_audit.csv", index=False)
print(out.to_string(index=False))

                                  metrik                                                            nilai
                               file_name                                PhiUSIIL_Phishing_URL_Dataset.csv
                         file_size_bytes                                                         56854345
                             file_sha256 a236549cd369cd80bd478ff8e1779cbf44c58d5c3f79f7a51a1adbed7d06d1c6
                           download_date                                                       2026-11-12
                                  n_rows                                                           235795
                                  n_cols                                                               56
                            n_legitimate                                                           134850
                              n_phishing                                                           100945
                          phishing_ratio      

## 8. Temuan dan keputusan yang perlu dicatat kelompok
Isi bagian ini setelah membaca output di atas (jangan sampai melihat hasil test final dulu):

- Apakah angka audit (baris, kolom, distribusi kelas) sesuai dengan proposal?
- Bagaimana kebijakan URL berulang dan konflik label?
- Kebijakan registered domain: default atau private PSL? Alasannya?
- Apakah klasifikasi F-URL / F-Page / Belum jelas disetujui Dika dan dosen?
- Apakah ada fold grouped yang tidak seimbang dan perlu ditangani?